[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/02_mathematics_for_ml/07_likelihood_mle_map.ipynb)

# Bài 07 — Likelihood, MLE và MAP

**🎯 Mục tiêu**
- Phân biệt **probability** và **likelihood**.
- Ước lượng tham số bằng **Maximum Likelihood (MLE)** — cách nhiều model ML "được sinh ra".
- Thấy **MSE = MLE với nhiễu Gauss** và **cross-entropy = MLE với Bernoulli**.
- **MAP**: thêm prior → chính là **regularization** (L2 ↔ Gaussian prior, L1 ↔ Laplace prior).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/02_mathematics_for_ml

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(42)

## 1. Probability vs Likelihood

Cùng một công thức $p(\text{data} \mid \theta)$, nhưng hai cách nhìn:
- **Probability:** cố định $\theta$, hỏi dữ liệu nào có khả năng xảy ra.
- **Likelihood** $\mathcal L(\theta) = p(\text{data} \mid \theta)$: cố định **dữ liệu đã quan sát**, hỏi $\theta$ nào giải thích dữ liệu tốt nhất. Likelihood **không** phải phân phối xác suất theo $\theta$ (không cần tích phân bằng 1).

Ví dụ: tung một đồng xu không biết có cân bằng không, được 7 ngửa trong 10 lần. $\theta = P(\text{ngửa})$:

$$\mathcal L(\theta) = \theta^7(1-\theta)^3$$

In [ ]:
theta = np.linspace(0.001, 0.999, 500)
heads, n = 7, 10
likelihood = theta**heads * (1 - theta) ** (n - heads)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(theta, likelihood)
axes[0].axvline(heads / n, color="red", ls="--", label=f"θ_MLE = {heads / n}")
axes[0].set_xlabel("θ")
axes[0].set_title("Likelihood L(θ)")
axes[0].legend()
axes[1].plot(theta, np.log(likelihood))
axes[1].axvline(heads / n, color="red", ls="--")
axes[1].set_xlabel("θ")
axes[1].set_title("Log-likelihood — cùng điểm cực đại, dễ tính hơn")
plt.show()

## 2. Maximum Likelihood Estimation (MLE)

$$\hat\theta_{MLE} = \arg\max_\theta \prod_{i=1}^n p(x^{(i)} \mid \theta) = \arg\max_\theta \sum_{i=1}^n \log p(x^{(i)} \mid \theta)$$

Vì sao lấy **log**? (1) tích của nhiều số nhỏ → underflow, tổng log thì không; (2) đạo hàm của tổng dễ hơn đạo hàm của tích; (3) log đồng biến nên không đổi điểm cực đại. Trong ML ta thường **cực tiểu negative log-likelihood (NLL)**.

In [ ]:
probs = rng.uniform(0.1, 0.9, 2000)
print("Tích 2000 xác suất:", np.prod(probs), "← underflow về 0!")
print("Tổng log           :", np.sum(np.log(probs)))

### 2.1 MLE cho Bernoulli → cross-entropy

$n$ quan sát $y^{(i)} \in \{0, 1\}$, $P(y = 1) = \theta$. Log-likelihood:

$$\log\mathcal L(\theta) = \sum_i \left[y^{(i)}\log\theta + (1 - y^{(i)})\log(1-\theta)\right]$$

Đạo hàm bằng 0 ⇒ $\hat\theta = \frac{1}{n}\sum_i y^{(i)}$ (tỉ lệ số lần ngửa — đúng như trực giác).

**Điểm mấu chốt:** nếu $\theta$ phụ thuộc vào $\mathbf x$ qua $\theta = \sigma(\mathbf w^\top\mathbf x + b)$, thì $-\frac1n\log\mathcal L$ chính là **binary cross-entropy** — loss của Logistic Regression (chương 07).

### 2.2 MLE cho phân phối chuẩn

Với $x^{(i)} \sim \mathcal N(\mu, \sigma^2)$: $\hat\mu_{MLE} = \bar x$, $\hat\sigma^2_{MLE} = \frac1n\sum_i(x^{(i)} - \bar x)^2$ (chia $n$, không phải $n-1$ → hơi chệch, bài 08).

In [ ]:
data = rng.normal(loc=5.0, scale=2.0, size=200)
mu_hat, sigma_hat = data.mean(), data.std()          # std mặc định ddof=0 → MLE
print(f"MLE: μ = {mu_hat:.3f}, σ = {sigma_hat:.3f}  (thật: 5.0, 2.0)")

# Kiểm chứng bằng tối ưu số: cực tiểu NLL
from scipy.optimize import minimize

nll = lambda params: -np.sum(stats.norm(params[0], np.exp(params[1])).logpdf(data))
res = minimize(nll, x0=[0.0, 0.0])
print(f"Tối ưu số: μ = {res.x[0]:.3f}, σ = {np.exp(res.x[1]):.3f}")

### 2.3 MLE với nhiễu Gauss = Least Squares (MSE)

Giả sử $y^{(i)} = \mathbf w^\top\mathbf x^{(i)} + \varepsilon^{(i)}$ với $\varepsilon^{(i)} \sim \mathcal N(0, \sigma^2)$ độc lập. Khi đó:

$$-\log\mathcal L(\mathbf w) = \frac{1}{2\sigma^2}\sum_i\left(y^{(i)} - \mathbf w^\top\mathbf x^{(i)}\right)^2 + \text{hằng số}$$

⇒ **cực đại likelihood ⇔ cực tiểu tổng bình phương sai số**. MSE không phải lựa chọn tuỳ tiện — nó là hệ quả của giả định nhiễu Gauss. Nếu nhiễu có đuôi dày (nhiều outlier), giả định này sai → nên dùng loss khác (MAE ↔ nhiễu Laplace, Huber...).

In [ ]:
x = rng.uniform(-3, 3, 100)
y = 2.0 * x + 1.0 + rng.normal(0, 1.0, 100)
Xb = np.c_[np.ones_like(x), x]

w_ls, *_ = np.linalg.lstsq(Xb, y, rcond=None)                                   # least squares
nll_w = lambda w: -np.sum(stats.norm(Xb @ w, 1.0).logpdf(y))                    # NLL với nhiễu Gauss
w_mle = minimize(nll_w, x0=[0.0, 0.0]).x
print("Least squares:", w_ls.round(4))
print("MLE (Gauss)  :", w_mle.round(4))

## 3. MAP — Maximum A Posteriori

MLE chỉ dựa vào dữ liệu. Với **ít dữ liệu**, MLE có thể cực đoan: tung đồng xu 3 lần được 3 ngửa ⇒ $\hat\theta_{MLE} = 1$ ("đồng xu không bao giờ ra sấp"!). MAP thêm **prior** $p(\theta)$ — niềm tin trước khi thấy dữ liệu:

$$\hat\theta_{MAP} = \arg\max_\theta\; p(\theta \mid \text{data}) = \arg\max_\theta\; \big[\log p(\text{data} \mid \theta) + \log p(\theta)\big]$$

Với đồng xu, prior Beta($a, b$) cho nghiệm: $\hat\theta_{MAP} = \frac{\#\text{ngửa} + a - 1}{n + a + b - 2}$ — giống như **cộng thêm các lần tung "ảo"**. Đây cũng chính là **Laplace smoothing** của Naive Bayes (chương 09).

In [ ]:
heads, n = 3, 3
a, b = 5, 5          # prior: tin rằng đồng xu gần cân bằng
theta = np.linspace(0.001, 0.999, 500)
prior = stats.beta(a, b).pdf(theta)
like = theta**heads * (1 - theta) ** (n - heads)
posterior = prior * like
posterior /= np.trapezoid(posterior, theta)

fig, ax = plt.subplots()
ax.plot(theta, prior, label="prior Beta(5, 5)")
ax.plot(theta, like / np.trapezoid(like, theta), label="likelihood (chuẩn hoá)")
ax.plot(theta, posterior, label="posterior", lw=2.5)
ax.axvline(heads / n, color="gray", ls=":", label=f"MLE = {heads / n:.2f}")
ax.axvline((heads + a - 1) / (n + a + b - 2), color="red", ls="--", label=f"MAP = {(heads + a - 1) / (n + a + b - 2):.2f}")
ax.legend(fontsize=9)
ax.set_title("3 lần tung đều ngửa: MAP kéo ước lượng về phía prior")
plt.show()

### MAP = Regularization

Với Linear Regression, đặt prior Gauss lên trọng số $w_j \sim \mathcal N(0, \tau^2)$:

$$-\log p(\mathbf w \mid \text{data}) = \underbrace{\frac{1}{2\sigma^2}\sum_i (y^{(i)} - \mathbf w^\top\mathbf x^{(i)})^2}_{\text{MSE}} + \underbrace{\frac{1}{2\tau^2}\lVert\mathbf w\rVert_2^2}_{\text{phạt L2}} + \text{const}$$

⇒ **MAP với prior Gauss = Ridge regression** ($\lambda = \sigma^2/\tau^2$). Prior Laplace ⇒ **Lasso** (phạt L1). Prior càng "chặt" ($\tau$ nhỏ) → regularization càng mạnh.

| Góc nhìn tối ưu | Góc nhìn xác suất |
|---|---|
| MSE loss | MLE + nhiễu Gauss |
| MAE loss | MLE + nhiễu Laplace |
| Cross-entropy | MLE + Bernoulli/Categorical |
| + L2 penalty (Ridge, weight decay) | MAP + prior Gauss |
| + L1 penalty (Lasso) | MAP + prior Laplace |

In [ ]:
x_small = rng.uniform(-3, 3, 8)                       # rất ít dữ liệu
y_small = 2.0 * x_small + 1.0 + rng.normal(0, 1.0, 8)
Xs = np.c_[np.ones_like(x_small), x_small, x_small**2, x_small**3, x_small**4, x_small**5]   # model quá phức tạp

for tau in [np.inf, 1.0, 0.1]:
    lam = 0.0 if np.isinf(tau) else 1.0 / tau**2
    penalty = np.diag([0.0] + [lam] * (Xs.shape[1] - 1))          # không phạt bias
    w = np.linalg.solve(Xs.T @ Xs + penalty, Xs.T @ y_small)
    name = "MLE (không prior)" if np.isinf(tau) else f"MAP, τ = {tau}"
    print(f"{name:20s} ||w|| = {np.linalg.norm(w[1:]):8.3f}   w = {w.round(2)}")

Prior càng chặt → trọng số càng nhỏ → model càng "đơn giản" → giảm overfitting (chương 05, 06).

## 4. Bayesian đầy đủ (giới thiệu)

MAP vẫn chỉ trả về **một** giá trị $\theta$. Cách tiếp cận Bayesian đầy đủ giữ **toàn bộ phân phối posterior** → biết được **mức độ không chắc chắn** của dự đoán. Chương 23 (Bayesian linear regression, Gaussian process) và chương 37 (uncertainty) đi sâu vào hướng này.

## 🧠 Kiểm tra nhanh

1. Tung xúc xắc 60 lần, mặt 6 xuất hiện 15 lần. MLE của $P(6)$ là bao nhiêu?
2. Vì sao luôn tối ưu **log**-likelihood thay vì likelihood?
3. Giả định nào về nhiễu khiến MLE trở thành bài toán least squares?
4. Ridge regression tương ứng với prior gì trên trọng số? Lasso thì sao?

## 📌 Tóm tắt

- Likelihood = xác suất của dữ liệu đã quan sát, xem như hàm của tham số.
- MLE: cực đại log-likelihood ⇔ cực tiểu NLL; MSE và cross-entropy đều là NLL.
- MAP = MLE + log prior ⇔ loss + regularization; hữu ích khi ít dữ liệu.

## 📚 Tài liệu tham khảo

- Bishop — *Pattern Recognition and Machine Learning*, chương 1.2, 3.3
- Vũ Hữu Tiệp — [MLE và MAP](https://machinelearningcoban.com/2017/07/17/mlemap/)
- Kevin Murphy — *Probabilistic Machine Learning: An Introduction* (2022), chương 4